In [11]:
API_KEY = ""   # TODO: Replace with your Youtube API Key
ACCESS_TOKEN = ""  # TODO: Replace with your Youtube OAuth client secret

In [12]:
%pip install -q -U nflx-genai
%pip install -q -U langchain
%pip install -q -U  arxiv
%pip install -U langchain-openai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 1.1.10 requires openai<3.0.0,>=2.20.0, but you have openai 1.109.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.netflix.net/simple
  Using cached https://pypi.netflix.net/packages/19935573881/openai-2.24.0-py3-none-any.whl.metadata (

In [13]:
import requests
import nflx_genai as genai
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import arxiv
import ast
import json

In [14]:
llm = ChatOpenAI(model="gpt-4o")
arxiv_client = arxiv.Client()

In [15]:
GOAL_PROMPT = f"""
You are passionate educator and you are the best in your domain. 
Till now you have been helping people offline but now you want to 
help a wider audience via Youtube platform. In order to help your 
target audience. I will provide you the data that I scrapped from 
Youtube comment section. My aim was to first make videos about the 
topic that people actually want to see. 
Therefor, I will provide you with the topic I am want to make videos
about. Searches for various youtube videos given the topic, 
this will you the list of youtube video ids. 
Then using this list of video ids fetch all the comments related to 
those videos. Thereafter, filter out the comments that have some suggestions 
around what audience would like to hear more about. Combine comments 
before filtering them otherwise you will be wasting a lot of calls. 
Using ONLY the relevant comments, find the topics themes that these comments point to. 
There after query arxiv to suggest the research papers related to 
these topics. The idea is that you need some resources to dive deep about these topic themes.
Your objective is figure out top 3-5 ideas along with the corresponding arxiv papers 
so that I can plan and deliver the best videos in next 2 months. 
The search keyword is:
"""

RELEVANT_COMMENTS_PROMPT = f"""
    You are passionate educator and you are the best in your domain. 
    Till now you have been helping people offline but now you want to 
    help a wider audience via Youtube platform. In order to help your 
    target audience. I will provide you the comments that I scrapped from 
    Youtube comment section. Identify and extract ONLY the comments 
    that contain genuine suggestions, feature requests, or ideas for 
    future content from the audience that I can make videos about.
    
    Format your response as a simple list of the relevant comments, 
    each separated by '|||'. If none are relevant, reply with: NO_RELEVANT_COMMENTS
    
    Comments:
    """

FIND_TOPIC_PROMPT = f"""
    You are passionate educator and you are the best in your domain. 
    Till now you have been helping people offline but now you want to 
    help a wider audience via Youtube platform. In order to help your 
    target audience. I have used an AI agent to take the comment section
    and identify research areas. Now, I want you to identify the top 5 
    research themes from these research areas such that I can query arxiv 
    using these themes. Research Areas Identified: 
"""
    

In [16]:
def invoke_get_request(url):
  response = requests.get(url)
  return response.json()

In [22]:
def build_search_url(query):
  return f"https://www.googleapis.com/youtube/v3/search?q={query}&type=video&part=snippet&key=" + API_KEY + "&access_token=" + ACCESS_TOKEN

def build_list_threads_using_video_id_url(video_id):
  return f"https://www.googleapis.com/youtube/v3/commentThreads?videoId={video_id}&part=id,replies,snippet&textFormat=plainText&key=" + API_KEY + "&access_token=" + ACCESS_TOKEN

@tool
def fetch_youtube_video_ids_with_keywords(keyword="Artificial intelligence"):
    """fetches the youtube video ids relevant to the keyword passed as input"""
    # Code to figure out data for videos which have a KEYWORD in them
    search_video_results = invoke_get_request(build_search_url(keyword))
    print(search_video_results)

    # Code to figure out videoIds from the json object received.
    parsed_video_ids = []
    for item in search_video_results['items']:
      parsed_video_ids.append(item['id']['videoId'])
    
    # FIX: If the LLM sends a string that looks like a list (due to JSON errors)
    if isinstance(parsed_video_ids, str):
        try:
            # Safely evaluate string representation of a list
            parsed_video_ids = ast.literal_eval(parsed_video_ids)
        except:
            # Fallback: if it's just one ID or comma-separated
            parsed_video_ids = [i.strip() for i in parsed_video_ids.replace("[","").replace("]","").replace("'","").split(",")]
    return parsed_video_ids

@tool
def fetch_youtube_comments_given_video_ids(parsed_video_ids):
    """Fetches Youtube comments given a list of video ids"""
    comment_collection = []
    for video_id in parsed_video_ids:
      raw_comments = invoke_get_request(build_list_threads_using_video_id_url(video_id))
    
      if 'items' not in raw_comments: # These videos had comments turned off
        continue

      for item in raw_comments['items']:
        comment_collection.append(item['snippet']['topLevelComment']['snippet']['textDisplay'])
    
        if 'replies' in item:
          for reply in item['replies']['comments']:
            comment_collection.append(reply['snippet']['textDisplay'])
    return comment_collection

@tool
def find_relevant_comments(comment_list, keyword):
    """
    Optimized Tool: Processes a batch of comments at once to filter for relevance.
    """
    formatted_comments = "\n".join([f"{i}. {c}" for i, c in enumerate(comment_list)])
    
    response = invoke_llm(RELEVANT_COMMENTS_PROMPT + formatted_comments)
    
    if not response or "NO_RELEVANT_COMMENTS" in response:
        return []
    
    relevant_list = [c.strip() for c in response.split("|||") if c.strip()]
    return relevant_list

@tool
def find_topics_themes(comment_collection):
    """Find Topics: Given the list of relevant comments this tool can find the topics or themes the comments are about."""
    text_input = "\n".join(comment_collection) if isinstance(comment_collection, list) else comment_collection
    
    filtered_topics = invoke_llm(FIND_TOPIC_PROMPT + text_input)
    
    #print("Focus Topics Identified:".upper(), filtered_topics)
    return filtered_topics
    
@tool
def query_arxiv(exact_topics):
    """Query arxiv:This tool fetches research papers given a list of exact topics"""
    if exact_topics is None or len(exact_topics) == 0:
        print("No exact topics shared to query arxiv")
        return

    all_papers = []
    
    if isinstance(exact_topics, str):
        exact_topics = [t.strip() for t in exact_topics.split(",")]
        
    for topic in exact_topics: 
        search = arxiv.Search(
        query = topic,
          max_results = 2,
          sort_by = arxiv.SortCriterion.SubmittedDate  # can be relevance too
        )
        #print("For the topics: ", topic)
        research_paper_for_topic = list(arxiv_client.results(search))
        topic_results = []
        for paper in research_paper_for_topic:
            topic_results.append({
                "title": paper.title,
                "url": paper.entry_id,
            })
        all_papers.append({topic: topic_results})
    #print("PAPERS suggested from arxiv: ".upper(), all_papers)
    return all_papers

In [18]:
# Pass the input as a dictionary with the key "exact_topics"
# test_result = query_arxiv.invoke({"exact_topics": ["Artificial Intelligence", "Quantum Computing"]})

# # Print the result to verify the links are working
# import json
# print(json.dumps(test_result, indent=2))

In [19]:
#Code to invoke llm
def invoke_llm(prompt):
    try: 
        completion = llm.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "user", "content": prompt}, #.join(parameters)
            ]
        )
        return completion.choices[0].message.content
    except Exception as e:
      print(e)

In [20]:
tools = [fetch_youtube_video_ids_with_keywords, fetch_youtube_comments_given_video_ids, find_relevant_comments, find_topics_themes, query_arxiv]
tool_dictionary = {
    "fetch_youtube_video_ids_with_keywords": fetch_youtube_video_ids_with_keywords,
    "fetch_youtube_comments_given_video_ids": fetch_youtube_comments_given_video_ids,
    "find_relevant_comments": find_relevant_comments,
    "find_topics_themes": find_topics_themes,
    "query_arxiv": query_arxiv
}
llm_with_tools = llm.bind_tools(tools)

In [ ]:
while True:
    user_input = input("👤 You: ")
    if user_input.lower() == "quit":
        print("🤖 Agent: Goodbye!")
        break
    messages = []
    messages.append(HumanMessage(content=GOAL_PROMPT + user_input))
    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if response.tool_calls:
            tool_call = response.tool_calls[0]
            tool_name = tool_call["name"]
            args = tool_call["args"]
    
            print(f"🛠 Calling tool: {tool_name}")
            
            messages.append(
                ToolMessage(
                    tool_call_id=tool_call["id"], 
                    content=str(tool_dictionary[tool_name].invoke(args)),
                    name=tool_name
                ))
            continue 
        else:
            print("\n--- Research Insights ---")
            print("🤖 Agent:",json.dumps(response.content, indent=4))
            print("--------------------------------------------------\n")
            break


👤 You:  Quantum Physics


🛠 Calling tool: fetch_youtube_video_ids_with_keywords
PARSED VIDEO IDs:  ['BHEhxPuMmQI', 't06aTX9jM34', 'p9pPjASnnxw', 'K5Po5R-1rgY', 'gD7spIB_Q14']
🛠 Calling tool: fetch_youtube_comments_given_video_ids
🛠 Calling tool: find_relevant_comments
🛠 Calling tool: find_topics_themes
🛠 Calling tool: query_arxiv

--- Research Insights ---
🤖 Agent: "Here are the top themes and corresponding research papers from arXiv that align with your audience's interests and could serve as excellent resources for creating insightful YouTube content:\n\n1. **Quantum Entanglement and Supercomputing**:\n   - [Geometric Resilience of Quantum LiDAR in Turbulent Media: A Wasserstein Distance Approach](http://arxiv.org/abs/2602.24280v1)\n   - [Asymptotically Solvable Quantum Circuits](http://arxiv.org/abs/2602.24276v1)\n\n2. **Time Travel Paradoxes and Quantum Mechanics**:\n   - [UFO-4D: Unposed Feedforward 4D Reconstruction from Two Images](http://arxiv.org/abs/2602.24290v1)\n   - [Unfolding without Iterations, A